# Plot original scaling from checkpoints

This notebook does not run the NCC search. It only loads precomputed checkpoint/result files from `data/` and plots them.

In [1]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

%config InlineBackend.figure_format = 'retina'
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams.update({"font.size": 18})
plt.rcParams["text.usetex"] = True

SAVE_PDF = False
FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_STEM = "NCC_original_mean_r"


def fit_power_law(x, y):
    coeffs = np.polyfit(np.log(x), np.log(y), 1)
    slope = coeffs[0]
    prefactor = math.exp(coeffs[1])
    return slope, prefactor


def scaled_reference(x, y0, x0, exponent):
    return y0 * (x / x0) ** exponent


def format_tick_labels(values):
    values = np.array(values, dtype=float)
    if np.allclose(values, np.round(values)):
        return [f"{int(round(val))}" for val in values]
    return [f"{val:.4f}".rstrip("0").rstrip(".") for val in values]


def plot_panel(ax, x, y, expected_exp, xlabel, title, invert_x=False, x_tick_labels=None):
    slope, prefactor = fit_power_law(np.array(x, dtype=float), np.array(y, dtype=float))
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    fit = prefactor * x**slope

    ax.loglog(x, y, "o", ms=8, color="C0", label="data")
    ref = scaled_reference(x, y[0], x[0], expected_exp)
    ax.loglog(x, ref, "--", lw=2.0, color="C1", label=rf"theory slope ${expected_exp:.3f}$")
    ax.loglog(x, fit, ":", lw=2.2, color="C2", label=rf"fit slope ${slope:.3f}$")
    ax.xaxis.set_major_locator(mticker.FixedLocator(x))
    if x_tick_labels is None:
        x_tick_labels = format_tick_labels(x)
    ax.xaxis.set_major_formatter(mticker.FixedFormatter(x_tick_labels))
    ax.yaxis.set_major_locator(mticker.FixedLocator(y))
    ax.yaxis.set_major_formatter(mticker.FixedFormatter(format_tick_labels(y)))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())
    if invert_x:
        ax.invert_xaxis()
    ax.set_xlabel(xlabel, fontsize=24)
    ax.set_ylabel("min gate count", fontsize=24)
    ax.set_title(title, fontsize=24)
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=True, fontsize=21)
    return slope


def resolve_stem(data_dir, preferred_stem, pattern):
    preferred_json = data_dir / f"{preferred_stem}.json"
    preferred_npz = data_dir / f"{preferred_stem}.npz"
    if preferred_json.exists() and preferred_npz.exists():
        return preferred_stem
    candidates = sorted(data_dir.glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No checkpoint files found in {data_dir} matching {pattern}")
    return candidates[-1].stem


def load_checkpoint(data_dir=DATA_DIR, preferred_stem=DATA_STEM):
    stem = resolve_stem(data_dir, preferred_stem, "NCC_original_mean_r*.json")
    payload = json.loads((data_dir / f"{stem}.json").read_text())
    arrays = np.load(data_dir / f"{stem}.npz")
    print(f"loaded checkpoint: {stem}")
    return stem, payload, arrays


DATA_STEM, payload, arrays = load_checkpoint()
n_values = arrays["n_values"]
n_gate = arrays["n_gate"]
t_values = arrays["t_values"]
t_gate = arrays["t_gate"]
eps_values = arrays["eps_values"]
eps_gate = arrays["eps_gate"]
fits = payload["fits"]
payload["params"]


FileNotFoundError: No checkpoint files found in data matching NCC_original_mean_r*.json

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
slope_n = plot_panel(
    ax,
    n_values,
    n_gate,
    5 / 3,
    r"qubit number $N$",
    r"Trotter-LCU gate complexity versus qubit number $N$",
)
fig.tight_layout()
if SAVE_PDF:
    fig.savefig(FIG_DIR / "plot_original_scaling_n.pdf", bbox_inches="tight")
print(f"fitted slope (N): {slope_n:.3f}")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
slope_t = plot_panel(
    ax,
    t_values,
    t_gate,
    4 / 3,
    r"evolution time $T$",
    r"Trotter-LCU gate complexity versus evolution time $T$",
)
fig.tight_layout()
if SAVE_PDF:
    fig.savefig(FIG_DIR / "plot_original_scaling_t.pdf", bbox_inches="tight")
print(f"fitted slope (T): {slope_t:.3f}")
plt.show()


In [ ]:
precision_tick_labels = [f"{val:.4f}" for val in eps_values]

fig, ax = plt.subplots(figsize=(10, 6))
slope_eps = plot_panel(
    ax,
    eps_values,
    eps_gate,
    -1 / 3,
    r"target precision $\epsilon$",
    r"Trotter-LCU gate complexity versus target precision $\epsilon$",
    invert_x=True,
    x_tick_labels=precision_tick_labels,
)
fig.tight_layout()
if SAVE_PDF:
    fig.savefig(FIG_DIR / "plot_original_scaling_eps.pdf", bbox_inches="tight")
print(f"fitted slope (eps): {slope_eps:.3f}")
plt.show()
